# Tratando Enums do Python

Muitas vezes usamos Enums nos modelos para restringir campos a certos valores.

Por exemplo, um campo que deve conter uma cor de uma lista predefinida.

No código Python, não queremos usar literais de string soltos — isso favorece erros (de digitação) — então usamos enumerações.

In [1]:
from enum import Enum

In [2]:
class Color(Enum):
    red = "Red"
    green = "Green"
    blue = "Blue"
    orange = "Orange"
    yellow = "Yellow"
    cyan = "Cyan"
    white = "White"
    black = "Black"

No código, usamos as cores pelos membros da enumeração:

In [3]:
print(f"Color.red: {Color.red!r}")

Color.red: <Color.red: 'Red'>


E recuperamos o valor associado:

In [4]:
print(f"Color.orange.value: {Color.orange.value!r}")

Color.orange.value: 'Orange'


Suponha um endpoint de API em que o corpo da requisição exige uma dessas cores.

Queremos usar a enumeração no modelo para restringir os valores aceitos **e** mapeá-los de volta para a enumeração na desserialização.

Basta usar a enumeração como tipo do campo:

In [5]:
from pydantic import BaseModel, ConfigDict, ValidationError

In [6]:
class Model(BaseModel):
    color: Color

Desserializando com objetos Python:

In [7]:
print(f"Model(color=Color.red): {Model(color=Color.red)!r}")

Model(color=Color.red): Model(color=<Color.red: 'Red'>)


E desserializando uma string JSON?

In [8]:
data = """
{
    "color": "Red"
}
"""

print(f"Model.model_validate_json(data): {Model.model_validate_json(data)!r}")

Model.model_validate_json(data): Model(color=<Color.red: 'Red'>)


E se escolhermos um valor que não está na enumeração:

In [9]:
data = """
{
    "color": "Magenta"
}
"""

try:
    Model.model_validate_json(data)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
color
  Input should be 'Red', 'Green', 'Blue', 'Orange', 'Yellow', 'Cyan', 'White' or 'Black' [type=enum, input_value='Magenta', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum


E a serialização? Serializando para dicionário Python, esperamos que o membro da enumeração seja mantido:

In [10]:
data = """
{
    "color": "Red"
}
"""

m = Model.model_validate_json(data)

In [11]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'color': <Color.red: 'Red'>}


E serializando para JSON:

In [12]:
print(f"m.model_dump_json(): {m.model_dump_json()!r}")

m.model_dump_json(): '{"color":"Red"}'


Então, ao desserializar, o campo é preenchido com um **membro** da enum.

Mas e se quisermos só o **valor** da enum, e não o membro? Ou seja, usar a enum apenas para limitar os valores aceitos e guardar os
valores como estão.

Fazemos isso com a configuração `use_enum_values` (padrão `False`).

In [13]:
class Model(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    color: Color

In [14]:
m = Model(color=Color.cyan)

In [15]:
print(f"m.color: {m.color!r}")

m.color: 'Cyan'


In [16]:
print(f"type(m.color): {type(m.color)!r}")

type(m.color): <class 'str'>


O campo agora é uma string, não um membro da enumeração.

E, claro, a serialização para dict também traz a string:

In [17]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'color': 'Cyan'}


Mas a validação continua valendo:

In [18]:
data = """
{
    "color": "Magenta"
}
"""

try:
    Model.model_validate_json(data)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for Model
color
  Input should be 'Red', 'Green', 'Blue', 'Orange', 'Yellow', 'Cyan', 'White' or 'Black' [type=enum, input_value='Magenta', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum


Há uma pegadinha, porém: quando definimos um valor default.

Vejamos:

In [19]:
class Model(BaseModel):
    model_config = ConfigDict(use_enum_values=True)

    color: Color = Color.red

Criando uma instância com o default — **deveria** ser a string `"Red"`:

In [20]:
m = Model()
print(f"m.color, type(m.color): {m.color, type(m.color)!r}")

m.color, type(m.color): (<Color.red: 'Red'>, <enum 'Color'>)


E não é nada do que esperávamos!

Se passamos um valor na desserialização, funciona como esperado:

In [21]:
m = Model(color=Color.red)
print(f"m.color, type(m.color): {m.color, type(m.color)!r}")

m.color, type(m.color): ('Red', <class 'str'>)


E com desserialização de JSON (sem o campo, cai no default):

In [22]:
data = """{}"""

m = Model.model_validate_json(data)
print(f"m.color, type(m.color): {m.color, type(m.color)!r}")

m.color, type(m.color): (<Color.red: 'Red'>, <enum 'Color'>)


O problema é um velho conhecido: defaults não são validados — são usados como estão (por padrão).

Corrigimos isso habilitando a validação de defaults no nível do modelo (depois veremos como fazer isso por campo, em vez de para todos).

In [23]:
class Model(BaseModel):
    model_config = ConfigDict(use_enum_values=True, validate_default=True)

    color: Color = Color.red

In [24]:
m = Model()
print(f"m.color, type(m.color): {m.color, type(m.color)!r}")

m.color, type(m.color): ('Red', <class 'str'>)


Agora o default também virou string.

Fica a observação para quando usar `use_enum_values`.

Há motivo para usar essa opção?

Na experiência do autor, muito pouco. Normalmente aparece quando queremos serializar nós mesmos o resultado de `model_dump()` para JSON:
sem o Pydantic fazendo essa serialização, um membro de Enum no dicionário precisaria de tratamento especial, já que enums não são
serializáveis por padrão com `json.dumps()`.

Fora isso (e há outras formas de lidar com o problema, como `model_dump(mode="json")`), não há grande uso para a opção — mas ela existe.

---
## Anotações extras (do material de estudo): Enums dinâmicas

> As seções abaixo não fazem parte das aulas originais: são anotações de estudo que vieram junto do material e foram mantidas e traduzidas.

### Lista → Enum

In [25]:
from enum import Enum

my_list = ["apple", "banana", "cherry"]

# Cria dinamicamente uma Enum chamada "Fruit", mapeando cada elemento para ele mesmo:
Fruit = Enum("Fruit", {item.upper(): item for item in my_list})

print(f"Fruit.APPLE: {Fruit.APPLE}")          # Fruit.APPLE
print(f"Fruit.APPLE.value: {Fruit.APPLE.value}")    # "apple"

Fruit.APPLE: Fruit.APPLE
Fruit.APPLE.value: apple


In [26]:
from pydantic import BaseModel, Field

class MyModel(BaseModel):
    fruit_type: Fruit = Field(..., description="Uma de: apple, banana, cherry.")

model = MyModel(fruit_type="banana")
print(f"model: {model}")
# fruit_type=<Fruit.BANANA: 'banana'>

model: fruit_type=<Fruit.BANANA: 'banana'>


### Usando `StrEnum`

In [27]:
from enum import StrEnum

my_list = ["apple", "banana", "cherry"]

Fruit = StrEnum("Fruit", {item.upper(): item for item in my_list})

print(f"isinstance(Fruit.APPLE, str): {isinstance(Fruit.APPLE, str)!r}")          # StrEnum: o próprio membro é uma str

isinstance(Fruit.APPLE, str): True


### Carregando de YAML

In [28]:
# Exemplo de arquivo YAML (myconfig.yaml):
# fruits:
#   - apple
#   - banana
#   - cherry

In [29]:
from enum import Enum
import yaml

with open('myconfig.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Extrai a lista do YAML
fruits_list = config["fruits"]  # ex.: ["apple", "banana", "cherry"]

# Cria dinamicamente uma Enum chamada "Fruit"
Fruit = Enum("Fruit", [(item.upper(), item) for item in fruits_list])

print(f"Fruit.APPLE: {Fruit.APPLE}")        # Fruit.APPLE
print(f"Fruit.APPLE.value: {Fruit.APPLE.value}")  # "apple"

Fruit.APPLE: Fruit.APPLE
Fruit.APPLE.value: apple


In [30]:
# Para nomes de membro em minúsculas, basta mudar a criação das tuplas:
Fruit = Enum("Fruit", [(item, item) for item in fruits_list])
print(f"list(Fruit): {list(Fruit)!r}")

list(Fruit): [<Fruit.apple: 'apple'>, <Fruit.banana: 'banana'>, <Fruit.cherry: 'cherry'>]


### Mais configurações no YAML

In [31]:
import yaml
from enum import Enum

with open("my_new_config.yaml") as f:
    config_dict = yaml.safe_load(f)

# Retira a lista de frutas
fruits_list = config_dict.pop("fruits")  # ["apple", "banana", "cherry"]

# Monta a Enum dinamicamente
FruitEnum = Enum(
    "FruitEnum",
    [(fruit.upper(), fruit) for fruit in fruits_list]
)
# => FruitEnum.APPLE ("apple"), FruitEnum.BANANA ("banana"), FruitEnum.CHERRY ("cherry")
print(f"FruitEnum.APPLE, FruitEnum.BANANA, FruitEnum.CHERRY: {FruitEnum.APPLE, FruitEnum.BANANA, FruitEnum.CHERRY!r}")

FruitEnum.APPLE, FruitEnum.BANANA, FruitEnum.CHERRY: (<FruitEnum.APPLE: 'apple'>, <FruitEnum.BANANA: 'banana'>, <FruitEnum.CHERRY: 'cherry'>)


In [32]:
from pydantic import BaseModel

# Com FruitEnum definida, podemos usá-la num modelo
class MyConfig(BaseModel):
    favorite_fruit: FruitEnum
    max_size: int

# Instancia o modelo com o restante da configuração
my_config = MyConfig(**config_dict)

print(f"my_config: {my_config}")
# favorite_fruit=<FruitEnum.APPLE: 'apple'> max_size=42   (o YAML define favorite_fruit: apple)

my_config: favorite_fruit=<FruitEnum.APPLE: 'apple'> max_size=42
